# 03 Feature Engineering

This notebook turns the raw readings into model-ready signals. The goal is to make the data easier to read for the model without hiding the original story inside a black box.

## Load Data and Set Up the Workspace

We start from the raw CSV again, then set up a few folders for saving a sample engineered dataset and the column definitions we create along the way. Think of this as laying out clean trays before sorting parts on a workbench.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_PATH = Path("../data/raw/air_quality_readings.csv")
PROCESSED_DIR = Path("../data/processed")
OUTPUTS_DIR = Path("../outputs")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

label_cols = [
    "label_respiratory_risk",
    "label_cardiovascular_risk",
    "label_vulnerable_alert",
    "label_outdoor_warning",
    "label_industrial_event",
]

raw_df = pd.read_csv(DATA_PATH)
raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], errors="coerce")
raw_df.head()

,reading_id,timestamp,year,month,hour,day_of_week,is_weekend,is_rush_hour,season,station_id,station_type,elevation_m,near_highway,near_industry,temp_c,humidity_pct,wind_speed_ms,wind_dir_deg,pressure_hpa,precipitation_mm,visibility_km,temp_inversion,pm25,pm10,no2,o3,so2,co,benzene,aqi,label_respiratory_risk,label_cardiovascular_risk,label_vulnerable_alert,label_outdoor_warning,label_industrial_event
0,RDG0003315,2021-12-24 18:00:00,2021,12,18,4,0,1,Winter,STN_URBAN_01,Urban,100.4,1,0,11.9,60.4,0.5,NaN,1016.5,1.0,14.7,0,58.3,92.0,61.9,33.2,0.7,6.01,3.24,152,1,1,1,1,0
1,RDG0009080,2023-09-19 21:00:00,2023,9,21,1,0,0,Fall,STN_SUBURB_01,Suburban,61.9,0,0,18.4,45.4,12.4,246.1,1017.4,0.0,6.2,0,8.5,16.2,63.6,8.9,13.2,0.59,5.00,35,0,0,0,0,0
2,RDG0005620,2022-08-28 11:00:00,2022,8,11,6,1,0,Summer,STN_SUBURB_01,Suburban,27.1,0,0,26.7,52.9,2.9,239.5,1010.6,0.0,4.3,0,13.2,23.0,16.5,74.8,3.3,1.61,2.55,53,0,0,1,0,0
3,RDG0009213,2023-10-05 06:00:00,2023,10,6,3,0,0,Fall,STN_URBAN_02,Urban,96.1,0,0,18.2,43.4,1.9,345.0,1018.5,0.0,12.4,0,20.1,41.5,58.1,17.0,22.5,1.50,NaN,68,0,0,1,0,0
4,RDG0009206,2023-10-04 05:00:00,2023,10,5,2,0,0,Fall,STN_INDUSTRIAL_01,Industrial,11.8,1,1,9.4,43.0,17.5,24.7,1007.1,10.0,3.6,0,25.5,48.2,66.4,30.8,33.9,3.31,6.20,79,0,0,1,0,1


## Clean Categories and Mark Missing Values

Before we build any fancy features, we fix the small things that can quietly throw a model off: category labels written in different styles and the four fields that have missing values baked into the problem.

In [2]:
def clean_air_quality_categories(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned["season"] = (
        cleaned["season"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"autumn": "fall"})
        .str.title()
    )
    cleaned["station_type"] = cleaned["station_type"].astype(str).str.strip().str.title()

    missing_columns = ["benzene", "visibility_km", "wind_dir_deg", "pressure_hpa"]
    for column in missing_columns:
        cleaned[f"{column}_missing"] = cleaned[column].isna().astype(int)

    return cleaned


cleaned_df = clean_air_quality_categories(raw_df)
cleaned_df[["season", "station_type", "benzene_missing", "visibility_km_missing", "wind_dir_deg_missing", "pressure_hpa_missing"]].head()

,season,station_type,benzene_missing,visibility_km_missing,wind_dir_deg_missing,pressure_hpa_missing
0,Winter,Urban,0,0,1,0
1,Fall,Suburban,0,0,0,0
2,Summer,Suburban,0,0,0,0
3,Fall,Urban,1,0,0,0
4,Fall,Industrial,0,0,0,0


## Add Circular Time and Wind Features

Angles are awkward for a model if we leave them as plain numbers. A reading at 359 degrees is almost the same as 1 degree, but a regular linear model would treat them like they are worlds apart. Sine and cosine fix that by wrapping the compass and clock into a circle the model can understand.

In [3]:
feature_df = cleaned_df.copy()

feature_df["hour_sin"] = np.sin(2 * np.pi * feature_df["hour"] / 24).round(4)
feature_df["hour_cos"] = np.cos(2 * np.pi * feature_df["hour"] / 24).round(4)
feature_df["month_sin"] = np.sin(2 * np.pi * feature_df["month"] / 12).round(4)
feature_df["month_cos"] = np.cos(2 * np.pi * feature_df["month"] / 12).round(4)

wind_rad = np.deg2rad(feature_df["wind_dir_deg"].fillna(180))
feature_df["wind_sin"] = np.sin(wind_rad).round(4)
feature_df["wind_cos"] = np.cos(wind_rad).round(4)

feature_df[["hour_sin", "hour_cos", "month_sin", "month_cos", "wind_sin", "wind_cos"]].head()

,hour_sin,hour_cos,month_sin,month_cos,wind_sin,wind_cos
0,-1.0000,-0.0000,-0.000,1.0,0.0000,-1.0000
1,-0.7071,0.7071,-1.000,-0.0,-0.9143,-0.4051
2,0.2588,-0.9659,-0.866,-0.5,-0.8616,-0.5075
3,1.0000,0.0000,-0.866,0.5,-0.2588,0.9659
4,0.9659,0.2588,-0.866,0.5,0.4179,0.9085


## Build Composite Pollution Features

Once the raw pollutants are in a better shape, we can combine them into a few higher-level signals. These are the kind of shortcuts a human analyst might use when skimming a report: a fine-particle ratio, a combined oxidant load, or a quick indicator that something industrial may be happening.

In [4]:
feature_df["pm_fine_ratio"] = (feature_df["pm25"] / feature_df["pm10"].clip(lower=0.1)).round(3)
feature_df["oxidant_load"] = (feature_df["o3"] + feature_df["no2"]).round(1)
feature_df["combustion_index"] = (feature_df["no2"] * 0.6 + feature_df["co"] * 10).round(2)
feature_df["industrial_signature"] = (feature_df["so2"] * 0.5 + feature_df["benzene"].fillna(0) * 3).round(2)
feature_df["aqi_excess_150"] = (feature_df["aqi"] - 150).clip(lower=0)
feature_df["aqi_excess_100"] = (feature_df["aqi"] - 100).clip(lower=0)

for column in ["pm25", "pm10", "no2", "so2", "co", "benzene", "aqi"]:
    feature_df[f"log_{column}"] = np.log1p(feature_df[column].fillna(0))

feature_df[["pm_fine_ratio", "oxidant_load", "combustion_index", "industrial_signature", "aqi_excess_150", "aqi_excess_100"]].head()

,pm_fine_ratio,oxidant_load,combustion_index,industrial_signature,aqi_excess_150,aqi_excess_100
0,0.634,95.1,97.24,10.07,2,52
1,0.525,72.5,44.06,21.60,0,0
2,0.574,91.3,26.00,9.30,0,0
3,0.484,75.1,49.86,11.25,0,0
4,0.529,97.2,72.94,35.55,0,0


## Add Weather and Station Context Features

A pollution reading makes more sense when we know the weather around it. Wind can spread pollution out, rain can wash it away, and a temperature inversion can trap it near the ground. Station location matters too, because an urban rush-hour reading is not the same thing as a rural morning reading.

In [5]:
feature_df["dispersion_index"] = (feature_df["wind_speed_ms"] / (1 + feature_df["temp_inversion"] * 3)).round(3)
feature_df["precipitation_flag"] = (feature_df["precipitation_mm"] > 0).astype(int)
feature_df["heavy_rain"] = (feature_df["precipitation_mm"] > 5).astype(int)
feature_df["heat_ozone_risk"] = ((feature_df["temp_c"] > 25) & (feature_df["o3"] > 100)).astype(int)
feature_df["inversion_severity"] = (
    feature_df["temp_inversion"]
    * (1 / feature_df["wind_speed_ms"].clip(lower=0.5))
    * (feature_df["humidity_pct"] / 100)
).round(3)
feature_df["haze_flag"] = (feature_df["visibility_km"].fillna(1.0) < 5).astype(int)
feature_df["urban_heat"] = ((feature_df["station_type"] == "Urban") & (feature_df["temp_c"] > 28)).astype(int)
feature_df["industrial_downwind"] = (
    (feature_df["near_industry"] == 1) &
    (feature_df["wind_dir_deg"].fillna(180).between(180, 315))
).astype(int)
feature_df["urban_rush"] = ((feature_df["station_type"] == "Urban") & (feature_df["is_rush_hour"] == 1)).astype(int)

feature_df[["dispersion_index", "precipitation_flag", "heavy_rain", "heat_ozone_risk", "inversion_severity", "haze_flag", "urban_heat", "industrial_downwind", "urban_rush"]].head()

,dispersion_index,precipitation_flag,heavy_rain,heat_ozone_risk,inversion_severity,haze_flag,urban_heat,industrial_downwind,urban_rush
0,0.5,1,0,0,0.0,0,0,0,1
1,12.4,0,0,0,0.0,0,0,0,0
2,2.9,0,0,0,0.0,1,0,0,0
3,1.9,0,0,0,0.0,0,0,0,0
4,17.5,1,1,0,0.0,1,0,0,0


## Add Rolling and Lag Features by Station

This is the piece that gives the model a short memory. A single spike matters, but a rising pattern over a few hours is usually more informative, the same way a doctor cares about a trend instead of one isolated number.

In [6]:
rolling_df = feature_df.sort_values(["station_id", "timestamp"]).copy()

for column in ["pm25", "no2", "aqi"]:
    rolling_df[f"{column}_roll3_mean"] = (
        rolling_df.groupby("station_id")[column]
        .transform(lambda values: values.rolling(3, min_periods=1).mean())
        .round(2)
    )
    rolling_df[f"{column}_lag1"] = rolling_df.groupby("station_id")[column].shift(1)
    rolling_df[f"{column}_trend"] = (rolling_df[column] - rolling_df[f"{column}_roll3_mean"]).round(2)

rolling_df["pm25_rising"] = (rolling_df["pm25_trend"] > 5).astype(int)
rolling_df[["station_id", "timestamp", "pm25_roll3_mean", "pm25_lag1", "pm25_trend", "pm25_rising"]].head(10)

,station_id,timestamp,pm25_roll3_mean,pm25_lag1,pm25_trend,pm25_rising
1088,STN_INDUSTRIAL_01,2021-01-01 15:00:00,23.70,NaN,0.00,0
1783,STN_INDUSTRIAL_01,2021-01-01 18:00:00,54.95,23.7,31.25,1
7745,STN_INDUSTRIAL_01,2021-01-03 04:00:00,45.60,86.2,-18.70,0
3406,STN_INDUSTRIAL_01,2021-01-04 10:00:00,49.07,26.9,-14.97,0
4027,STN_INDUSTRIAL_01,2021-01-04 12:00:00,47.33,34.1,33.67,1
192,STN_INDUSTRIAL_01,2021-01-05 04:00:00,76.67,81.0,38.23,1
178,STN_INDUSTRIAL_01,2021-01-05 14:00:00,84.83,114.9,-26.23,0
739,STN_INDUSTRIAL_01,2021-01-06 01:00:00,69.43,58.6,-34.63,0
7112,STN_INDUSTRIAL_01,2021-01-06 17:00:00,45.00,34.8,-3.40,0
5842,STN_INDUSTRIAL_01,2021-01-07 10:00:00,38.20,41.6,0.00,0


## Save a Sample Feature Definition

To keep this notebook aligned with future code in `src/`, we save a small engineered sample and the list of feature columns we created. That gives us a simple checkpoint we can compare against later.

In [7]:
feature_columns = [
    column
    for column in rolling_df.columns
    if column not in {"reading_id", "timestamp"}
]

sample_columns = [
    "reading_id",
    "timestamp",
    "station_id",
    "station_type",
    "season",
    "hour",
    "pm25",
    "pm10",
    "no2",
    "o3",
    "so2",
    "co",
    "benzene",
    "aqi",
    "pm_fine_ratio",
    "oxidant_load",
    "dispersion_index",
    "pm25_roll3_mean",
    "pm25_lag1",
    "pm25_trend",
    "pm25_rising",
]

feature_spec_path = OUTPUTS_DIR / "feature_engineering_columns.json"
sample_path = PROCESSED_DIR / "feature_engineering_sample.csv"

rolling_df.loc[:, sample_columns].head(100).to_csv(sample_path, index=False)
pd.Series(feature_columns, name="feature_columns").to_json(feature_spec_path, orient="values")

print(f"Saved sample engineered dataset to: {sample_path}")
print(f"Saved feature column list to: {feature_spec_path}")
print(f"Total engineered columns: {len(feature_columns)}")

Saved sample engineered dataset to: ..\data\processed\feature_engineering_sample.csv
Saved feature column list to: ..\outputs\feature_engineering_columns.json
Total engineered columns: 75
